In [1]:
import re
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup 

In [2]:
with open('vishal_new.html','r',encoding='utf-8') as f:
    res=f.read()
soup = BeautifulSoup(res,'html.parser')

# Notebook Overview
This notebook loads an HTML file, parses it with BeautifulSoup, and extracts mobile phone data from the page.
The extracted fields are:
- `mobilename`
- `star`
- `rating`
- `reviews`
- `display`
- `camera`
- `new_price`
- `old_price`
- `discount`


The fields extracted from the HTML page are:
- `mobilename`
- `star`
- `rating`
- `reviews`
- `display`
- `camera`
- `new_price`
- `old_price`
- `discount`

In [3]:
# Correct extraction logic for all fields using card wrappers
cards = soup.find_all('div', class_='_3pLy-c row')

mobilename = []
star = []
rating = []
reviews = []
display = []
camera = []
new_price = []
old_price = []
discount = []

for card in cards:
    name_tag = card.find('div', class_='_4rR01T')
    mobilename.append(' '.join(name_tag.text.split()) if name_tag else None)

    star_tag = card.find('div', class_='_3LWZlK')
    star.append(star_tag.text.strip() if star_tag else None)

    rr_tag = card.find('span', class_='_2_R_DZ')
    if rr_tag:
        text = rr_tag.text.replace('\n', ' ').replace('\xa0', ' ').strip()
        nums = re.findall(r'\d+', text.replace(',', ''))
        if len(nums) >= 2:
            rating.append(int(nums[0]))
            reviews.append(int(nums[1]))
        else:
            rating.append(None)
            reviews.append(None)
    else:
        rating.append(None)
        reviews.append(None)

    spec_tag = card.find('div', class_='fMghEO')
    if spec_tag:
        items = spec_tag.find('ul').find_all('li')
        display.append(items[1].text.strip() if len(items) > 1 else None)
        camera.append(items[2].text.strip() if len(items) > 2 else None)
    else:
        display.append(None)
        camera.append(None)

    np_tag = card.find('div', class_='_30jeq3 _1_WHN1')
    if np_tag:
        new_price.append(int(np_tag.text.strip().replace(',', '').replace('₹', '')))
    else:
        new_price.append(None)

    old_price_tag = card.find('div', class_='_3I9_wc _27UcVY')
    if old_price_tag:
        old_price.append(int(old_price_tag.text.strip().replace(',', '').replace('₹', '')))
    else:
        old_price.append(None)

    discount_tag = card.find('div', class_='_3Ay6Sb')
    if discount_tag:
        discount.append(int(discount_tag.text.strip().split('%')[0]))
    else:
        discount.append(None)


In [4]:
# Build a dictionary from the extracted lists
mobile_data = {
    'mobilename': mobilename,
    'star': star,
    'rating': rating,
    'reviews': reviews,
    'display': display,
    'camera': camera,
    'new_price': new_price,
    'old_price': old_price,
    'discount': discount
}

# Optionally convert the dictionary into a pandas DataFrame for easier viewing
mobile_df = pd.DataFrame(mobile_data)

print('Number of mobiles extracted:', len(mobile_df))
print(mobile_df.head())


Number of mobiles extracted: 24
                                          mobilename star  rating  reviews  \
0                   Huawei P9 (Mystic Silver, 32 GB)  4.5     829      265   
1                   Huawei P9 (Prestige Gold, 32 GB)  4.5     829      265   
2  SAMSUNG Galaxy S22 Plus 5G (Phantom Black, 128...  4.5    3234      364   
3  SAMSUNG Galaxy Z Flip3 5G (Phantom Black, 128 GB)  4.3    4262      285   
4  MOTOROLA Edge 30 Ultra (Interstellar Black, 25...  4.2     559       96   

                                 display  \
0    13.21 cm (5.2 inch) Full HD Display   
1    13.21 cm (5.2 inch) Full HD Display   
2   16.76 cm (6.6 inch) Full HD+ Display   
3   17.02 cm (6.7 inch) Full HD+ Display   
4  16.94 cm (6.67 inch) Full HD+ Display   

                                    camera  new_price  old_price  discount  
0           12MP + 12MP | 8MP Front Camera      39999    40999.0       2.0  
1           12MP + 12MP | 8MP Front Camera      39999    40999.0       2.0  
2  

In [5]:
# Diagnostic check: compare lengths of all extracted lists
fields = ['mobilename','star','rating','reviews','display','camera','new_price','old_price','discount']
for field in fields:
    value = globals().get(field)
    print(field, 'length =', len(value) if value is not None else 'missing')

print('\nSample values:')
print('mobilename', mobilename[:3])
print('rating', rating[:5])
print('reviews', reviews[:5])
print('old_price', old_price[:5])
print('discount', discount[:5])


mobilename length = 24
star length = 24
rating length = 24
reviews length = 24
display length = 24
camera length = 24
new_price length = 24
old_price length = 24
discount length = 24

Sample values:
mobilename ['Huawei P9 (Mystic Silver, 32 GB)', 'Huawei P9 (Prestige Gold, 32 GB)', 'SAMSUNG Galaxy S22 Plus 5G (Phantom Black, 128 GB)']
rating [829, 829, 3234, 4262, 559]
reviews [265, 265, 364, 285, 96]
old_price [40999, 40999, 101999, 95999, 74999]
discount [2, 2, 50, 53, 33]


In [6]:
# Inspect product item structure for consistent extraction
box = soup.find_all('div', class_='_4rR01T')
print('Product count by name:', len(box))
for i in range(min(3, len(box))):
    item = box[i]
    parent = item.parent
    grandparent = parent.parent
    print('\n=== Item', i, '===')
    print('name tag:', item.name)
    print('name class:', item.get('class'))
    print('parent tag:', parent.name, 'class:', parent.get('class'))
    print('grandparent tag:', grandparent.name, 'class:', grandparent.get('class'))
    print('parent text first 180 chars:', repr(item.parent.text.strip()[:180]))

Product count by name: 24

=== Item 0 ===
name tag: div
name class: ['_4rR01T']
parent tag: div class: ['col', 'col-7-12']
grandparent tag: div class: ['_3pLy-c', 'row']
parent text first 180 chars: 'Huawei P9 (Mystic Silver, 32 GB)\n\n4.5\n829\n\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\tRatings\xa0&\xa0265\n\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\tReviews\n\n\n3 GB RAM | 32 GB ROM | Expandable Upto\n\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t128 GB\n13.21 cm (5.2 inch'

=== Item 1 ===
name tag: div
name class: ['_4rR01T']
parent tag: div class: ['col', 'col-7-12']
grandparent tag: div class: ['_3pLy-c', 'row']
parent text first 180 chars: 'Huawei P9 (Prestige Gold, 32 GB)\n\n4.5\n829\n\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\tRatings\xa0&\xa0265\n\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\tReviews\n\n\n3 GB RAM | 32 GB ROM | Expandable Upto\n\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t128 GB\n13.21 cm (5.2 inch'

=== Item 2 ===
name tag: div
name class: ['_4rR01T']
parent tag: div class: ['col', 'col-7-12']
grandparent tag: div class: ['_3p

In [7]:
# Verify the count of full product cards
cards = soup.find_all('div', class_='_3pLy-c row')
print('Product card count by wrapper:', len(cards))
for i in range(min(3, len(cards))):
    card = cards[i]
    print('\n==== card', i, '====')
    print('card classes:', card.get('class'))
    print('card text includes:', repr(card.text.strip()[:120]))


Product card count by wrapper: 24

==== card 0 ====
card classes: ['_3pLy-c', 'row']
card text includes: 'Huawei P9 (Mystic Silver, 32 GB)\n\n4.5\n829\n\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\tRatings\xa0&\xa0265\n\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\tReviews\n\n\n3 GB RAM | 32 GB ROM'

==== card 1 ====
card classes: ['_3pLy-c', 'row']
card text includes: 'Huawei P9 (Prestige Gold, 32 GB)\n\n4.5\n829\n\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\tRatings\xa0&\xa0265\n\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\tReviews\n\n\n3 GB RAM | 32 GB ROM'

==== card 2 ====
card classes: ['_3pLy-c', 'row']
card text includes: 'SAMSUNG Galaxy S22 Plus 5G (Phantom Black, 128\n\t\t\t\t\t\t\t\t\t\t\t\t\t\tGB)\n\n4.5\n3,234\n\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\tRatings\xa0&\xa0364\n\t\t\t\t\t\t\t\t\t\t\t\t\t'


In [8]:
d={'mobilename':[],'star':[],'rating':[],'review':[],'display':[],'camera':[],'new_price':[],'old_price':[],'discount':[]}
box=soup.find_all("div",class_="_3pLy-c row")
for i in box:
    d['mobilename'].append(i.find('div',class_="_4rR01T").text.strip())
    d['star'].append(float(i.find('div',class_="_3LWZlK").text.strip()))
    # print("rating",i.find('span',class_="_2_R_DZ").text.strip().split("\xa0&\xa0")[0][:-8])
    # print("review",i.find('span',class_="_2_R_DZ").text.strip().split("\xa0&\xa0")[1][:-8])
    d['rating'].append(int(i.find('span',class_="_2_R_DZ").text.strip().split("\xa0&\xa0")[0][:-8].replace(",","")))
    d['review'].append(int(i.find('span',class_="_2_R_DZ").text.strip().split("\xa0&\xa0")[1][:-8].replace(",","")))
    d['display'].append(i.find("ul",class_="_1xgFaf").find_all("li")[1].text.strip())
    d['camera'].append(i.find("ul",class_="_1xgFaf").find_all("li")[2].text.strip())
    d['new_price'].append(i.find('div',class_="_30jeq3 _1_WHN1").text.strip()[1:])
    try:
        d['old_price'].append(i.find('div',class_="_3I9_wc _27UcVY").text.strip())
    except:
        d['old_price'].append(np.nan)
    try:
        d['discount'].append(i.find('div',class_="_3Ay6Sb").text.strip())
    except:
        d['discount'].append(np.nan)
df = pd.DataFrame(d)
df

,mobilename,star,rating,review,display,camera,new_price,old_price,discount
0,"Huawei P9 (Mystic Silver, 32 GB)",4.5,829,265,13.21 cm (5.2 inch) Full HD Display,12MP + 12MP | 8MP Front Camera,"39,999","₹40,999",2% off
1,"Huawei P9 (Prestige Gold, 32 GB)",4.5,829,265,13.21 cm (5.2 inch) Full HD Display,12MP + 12MP | 8MP Front Camera,"39,999","₹40,999",2% off
2,"SAMSUNG Galaxy S22 Plus 5G (Phantom Black, 128...",4.5,3234,364,16.76 cm (6.6 inch) Full HD+ Display,50MP + 12MP + 10MP | 10MP Front Camera,"49,999","₹1,01,999",50% off
3,"SAMSUNG Galaxy Z Flip3 5G (Phantom Black, 128\...",4.3,4262,285,17.02 cm (6.7 inch) Full HD+ Display,12MP + 12MP | 10MP Front Camera,"44,999","₹95,999",53% off
4,"MOTOROLA Edge 30 Ultra (Interstellar Black, 25...",4.2,559,96,16.94 cm (6.67 inch) Full HD+ Display,200MP + 50MP + 12MP | 60MP Front Camera,"49,999","₹74,999",33% off
5,"MOTOROLA Edge 30 Ultra (Starlight White, 256\n...",4.2,559,96,16.94 cm (6.67 inch) Full HD+ Display,200MP + 50MP + 12MP | 60MP Front Camera,"49,999","₹74,999",33% off
6,"SAMSUNG Galaxy A54 5G (Awesome Violet, 256 GB)",4.3,415,39,16.26 cm (6.4 inch) Full HD+ Display,50MP + 12MP + 5MP | 32MP Front Camera,"40,999","₹45,999",10% off
7,"OnePlus 8 (Glacial Green, 256 GB)",4.5,246,35,16.64 cm (6.55 inch) Display,48MP + 2MP + 16MP,"46,999","₹47,999",2% off
8,"SAMSUNG Galaxy S22 Plus 5G (Green, 128 GB)",4.5,3234,364,16.76 cm (6.6 inch) Full HD+ Display,50MP + 12MP + 10MP | 10MP Front Camera,"49,999","₹1,01,999",50% off
9,"OnePlus 8 (Interstellar Glow, 256 GB)",4.5,246,35,16.64 cm (6.55 inch) Display,48MP + 2MP + 16MP,"47,349","₹47,999",1% off


In [9]:
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
with open('html data/List of state and union territory capitals in India - Wikipedia (5_28_2025 8：46：50 AM).html','r',encoding = 'utf-8') as f:
    res = f.read()
soup = BeautifulSoup(res,'html.parser')
table = soup.find('table',class_ = 'wikitable sortable jquery-tablesorter')

head = table.find('thead')
col = []
for i in head.find_all('th'):
    col.append(i.text.strip())
col[5] = col[5].split('\n')[0]
col

body = table.find('tbody')
d = {col[0] : [],col[1] : [],col[2] : [],col[3] : [],col[4] : [],col[5] : []}
for i in body.find_all('tr'):
    l = i.find_all('td')
    d[col[0]].append(l[0].text.strip())
    d[col[1]].append(l[1].text.strip())
    d[col[2]].append(l[2].text.strip())
    d[col[3]].append(l[3].text.strip())
    d[col[4]].append(l[4].text.strip())
    d[col[5]].append(l[5].text.strip())


df = pd.DataFrame(d)


FileNotFoundError: [Errno 2] No such file or directory: 'html data/List of state and union territory capitals in India - Wikipedia (5_28_2025 8：46：50 AM).html'